# Build BM25 Index from TechQA Tokens

It builds the BM25 index from bm25_tokens.jsonl and saves:
- `/kaggle/working/bm25s_index/` - το bm25s index
- `/kaggle/working/bm25_index.pkl` - IDs, docs, metas

In [ ]:
!pip install -q bm25s

In [ ]:
import os

# Find the path to the dataset
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        fpath = os.path.join(root, f)
        size  = os.path.getsize(fpath) / (1024**3)
        print(f'{fpath}: {size:.2f} GB')

In [ ]:
# -- Path Setup ------------------------------------------------
# Adjust as needed based on the output above
TOKENS_PATH  = '/kaggle/input/datasets/yiannisman/bge-bm25-tokens-testing/bm25_tokens.jsonl'
BM25S_DIR    = '/kaggle/working/bm25s_index'
METADATA_PATH = '/kaggle/working/bm25_index.pkl'

print(f'Tokens file exists: {os.path.exists(TOKENS_PATH)}')
print(f'Size: {os.path.getsize(TOKENS_PATH)/1024**3:.2f} GB')

In [ ]:
import json
import pickle
import gc
import bm25s

REPORT_EVERY = 100_000

# -- Step 1: Loading ONLY the tokens -----------------------------------
print('Step 1: Loading tokens...')
all_tokens = []
total      = 0

with open(TOKENS_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        record = json.loads(line)
        all_tokens.append(record['tokens'])
        total += 1
        if total % REPORT_EVERY == 0:
            print(f'  ...{total} loaded tokens')

print(f'{total} documents loaded.')

In [ ]:
# -- Step 2: Building BM25 index -----------------
print('Step 2: Building BM25 index...')
retriever = bm25s.BM25()
retriever.index(all_tokens)
del all_tokens
gc.collect()
print('BM25 index was created!')

In [ ]:
# -- Step 3: Save index --------------------------------------------
print(f'Step 3: Save index -> {BM25S_DIR}...')
os.makedirs(BM25S_DIR, exist_ok=True)
retriever.save(BM25S_DIR, corpus=None)
del retriever
gc.collect()
print('Index saved!')

# Show files
for fname in os.listdir(BM25S_DIR):
    size = os.path.getsize(os.path.join(BM25S_DIR, fname)) / (1024**2)
    print(f'  {fname}: {size:.1f} MB')

In [ ]:
# -- Step 4: Uploading and storing IDs/docs/metas ---------------------
# Now the memory is free of tokens
print('Step 4: Uploading IDs, documents, and metadata...')
all_ids, all_docs, all_metas = [], [], []

with open(TOKENS_PATH, 'r', encoding='utf-8') as f:
    for i, line in enumerate(f):
        record = json.loads(line)
        all_ids.append(record['id'])
        all_docs.append(record['doc'])
        all_metas.append(record['meta'])
        if (i + 1) % REPORT_EVERY == 0:
            print(f'  ...{i+1} entries')

print(f'Saving metadata -> {METADATA_PATH}...')
with open(METADATA_PATH, 'wb') as f:
    pickle.dump({'ids': all_ids, 'docs': all_docs, 'metas': all_metas}, f)

print(f'Metadata saved!')
size = os.path.getsize(METADATA_PATH) / (1024**3)
print(f'   bm25_index.pkl: {size:.2f} GB')

In [ ]:
# -- Output confirmation ----------------------------------------------
print('\nFiles output:')
for root, dirs, files in os.walk('/kaggle/working'):
    for fname in files:
        fpath = os.path.join(root, fname)
        size  = os.path.getsize(fpath) / (1024**3)
        print(f'  {fpath}: {size:.3f} GB')

print()
print('='*60)
print('DONE! NEXT STEPS:')
print('1. Output -> Download bm25s_index/ and bm25_index.pkl')
print('2. docker cp bm25s_index knowledge_api:/data/bm25s_index')
print('3. docker cp bm25_index.pkl knowledge_api:/data/bm25_index.pkl')
print('4. docker restart knowledge_api')
print('='*60)